In [ ]:
eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
eval_trial_batch, target_lead_times=slice("6h", "168h"),
**dataclasses.asdict(task_config))  


In [ ]:
# Extract the variable of interest (e.g., '2m_temperature')
variable_name = '2m_temperature'
temperature_pred_old = predictions_old[variable_name]
temperature_pred_finetuned = predictions_finetuned[variable_name]
temperature_targets = eval_targets[variable_name]

# Define the time steps to visualize
time_steps = [0, 1]  # Adjust as needed

# Function to plot predictions over India
def plot_predictions(time_step):
    fig, axs = plt.subplots(1, 3, figsize=(18, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Old predictions
    ax = axs[0]
    temp_old = temperature_pred_old.isel(time=time_step).squeeze()
    temp_old.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='coolwarm',
        cbar_kwargs={'label': 'Temperature (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Old Predictions at Time Step {time_step}")

    # Finetuned predictions
    ax = axs[1]
    temp_finetuned = temperature_pred_finetuned.isel(time=time_step).squeeze()
    temp_finetuned.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='coolwarm',
        cbar_kwargs={'label': 'Temperature (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Finetuned Predictions at Time Step {time_step}")

    # Difference between finetuned and old predictions
    ax = axs[2]
    difference = temp_finetuned - temp_old
    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Difference at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

# Plot predictions for each time step
for time_step in time_steps:
    plot_predictions(time_step)

# Function to plot difference between finetuned predictions and ground truth
def plot_difference_with_targets(time_step):
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Difference between finetuned predictions and evaluation targets
    temp_finetuned = temperature_pred_finetuned.isel(time=time_step).squeeze()
    temp_targets = temperature_targets.isel(time=time_step).squeeze()
    difference = temp_finetuned - temp_targets

    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Finetuned Prediction vs Ground Truth at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

    return (difference)

# Function to plot difference between base predictions and ground truth
def plot_difference_with_targets_base(time_step):
    fig, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': ccrs.PlateCarree()})
    extent = [68, 98, 6, 38]  # Focus on India

    # Difference between finetuned predictions and evaluation targets
    temp_old = temperature_pred_old.isel(time=time_step).squeeze()
    temp_targets = temperature_targets.isel(time=time_step).squeeze()
    difference = temp_old - temp_targets

    difference.plot.pcolormesh(
        ax=ax, transform=ccrs.PlateCarree(), cmap='bwr',
        cbar_kwargs={'label': 'Temperature Difference (K)'})
    ax.set_extent(extent)
    ax.coastlines()
    ax.set_title(f"Base Prediction vs Ground Truth at Time Step {time_step}")

    plt.tight_layout()
    plt.show()

    return (difference)

# Plot differences with ground truth for each time step
differences_finetuned = []
differences_base = []
for time_step in time_steps:
    differences_finetuned.append(plot_difference_with_targets(time_step))
    differences_base.append(plot_difference_with_targets_base(time_step))